# Clase 3 · 01 · CAP y PACELC con tres réplicas

Antes de ver cada modelo NoSQL, simulamos el problema que todos enfrentan al distribuirse: **mantener copias de un dato en varios nodos**.

Guardamos el saldo de una billetera (`saldo`) en tres réplicas `A`, `B` y `C`. Vamos a cortar la red entre ellas y comparar dos políticas:

- **CP** (consistencia ante partición): sólo acepta lecturas y escrituras si llega a la mayoría (quórum). Si no, responde error.
- **AP** (disponibilidad ante partición): cada nodo responde con lo que tiene, aunque esté desactualizado. Al reparar la red se reconcilia con *last-write-wins* (gana la versión más nueva).

Este notebook es Python puro: no usa Spark ni tablas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


class ReplicatedStore:
    """Almacén clave-valor replicado de juguete. Cada réplica guarda (valor, versión)."""

    def __init__(self, nodes, mode):
        assert mode in ("CP", "AP")
        self.mode, self.nodes = mode, list(nodes)
        self.data = {n: {} for n in self.nodes}
        self.groups = [set(self.nodes)]          # grupos que se ven entre sí
        self.version = 0
        self.log = []

    def _group(self, via):
        return next(g for g in self.groups if via in g)

    def _has_quorum(self, via):
        return len(self._group(via)) >= len(self.nodes) // 2 + 1

    def _record(self, operation, via, result):
        row = {"paso": len(self.log) + 1, "operación": operation, "vía": via, "resultado": result}
        row.update({n: self.data[n].get("saldo", (None, 0))[0] for n in self.nodes})
        self.log.append(row)
        return result

    def partition(self, *groups):
        self.groups = [set(g) for g in groups]
        return self._record("partición " + " | ".join("".join(sorted(g)) for g in self.groups), "-", "red cortada")

    def heal(self):
        self.groups = [set(self.nodes)]
        for key in {k for d in self.data.values() for k in d}:
            winner = max((d[key] for d in self.data.values() if key in d), key=lambda vv: vv[1])
            for n in self.nodes:
                self.data[n][key] = winner
        return self._record("reparación (last-write-wins)", "-", "réplicas iguales")

    def write(self, key, value, via):
        if self.mode == "CP" and not self._has_quorum(via):
            return self._record(f"write {key}={value}", via, "ERROR: sin quórum")
        self.version += 1
        for n in self._group(via):
            self.data[n][key] = (value, self.version)
        return self._record(f"write {key}={value}", via, "OK")

    def read(self, key, via):
        if self.mode == "CP":
            if not self._has_quorum(via):
                return self._record(f"read {key}", via, "ERROR: no disponible")
            value = max((self.data[n].get(key, (None, 0)) for n in self._group(via)), key=lambda vv: vv[1])[0]
        else:
            value = self.data[via].get(key, (None, 0))[0]
        return self._record(f"read {key}", via, f"devuelve {value}")

## El escenario

1. Se escribe `saldo=100` con la red sana.
2. La red se parte: `A,B` quedan de un lado y `C` aislado.
3. Un cliente descuenta una compra en `A`: `saldo=80`.
4. Otro cliente lee el saldo en `C`.
5. Un tercer cliente escribe `saldo=50` en `C` (por ejemplo, otra compra).
6. Se repara la red.
7. Se lee el saldo final.

In [ ]:
def run_scenario(mode):
    store = ReplicatedStore(["A", "B", "C"], mode)
    store.write("saldo", 100, via="A")
    store.partition({"A", "B"}, {"C"})
    store.write("saldo", 80, via="A")
    store.read("saldo", via="C")
    store.write("saldo", 50, via="C")
    store.heal()
    store.read("saldo", via="B")
    return pd.DataFrame(store.log)

cp_log, ap_log = run_scenario("CP"), run_scenario("AP")
print("Modo CP"); display(cp_log)
print("Modo AP"); display(ap_log)

### Visualización: qué valor tiene cada réplica en cada paso

Cada celda muestra el saldo guardado por una réplica. El marco rojo marca respuestas de error; el naranja, lecturas que devolvieron un valor distinto del último escrito con éxito.

In [ ]:
def plot_replicas(log, title, ax):
    nodes = ["A", "B", "C"]
    values = log[nodes].to_numpy(dtype=float)
    ax.imshow(np.nan_to_num(values.T, nan=0), cmap="Blues", vmin=-40, vmax=250, aspect="auto")
    last_ok = None
    for i, row in log.iterrows():
        for j, n in enumerate(nodes):
            v = row[n]
            ax.text(i, j, "-" if pd.isna(v) else int(v), ha="center", va="center", fontsize=11)
        if row["operación"].startswith("write") and row["resultado"] == "OK":
            last_ok = int(row["operación"].split("=")[1])
        color = None
        if row["resultado"].startswith("ERROR"):
            color = "#dc2626"
        elif row["operación"].startswith("read") and row["resultado"].split()[-1] != str(last_ok):
            color = "#f59e0b"
        if color:
            ax.add_patch(plt.Rectangle((i - 0.47, -0.47), 0.94, 2.94, fill=False, edgecolor=color, lw=3))
    ax.set_yticks(range(3), [f"réplica {n}" for n in nodes])
    ax.set_xticks(range(len(log)), [f"{r.paso}. {r.operación}\n({r.vía}) {r.resultado}" for r in log.itertuples()],
                  rotation=35, ha="right", fontsize=8)
    ax.set_title(title)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
plot_replicas(cp_log, "CP: prefiere error antes que un dato viejo", axes[0])
plot_replicas(ap_log, "AP: siempre responde, aunque el dato esté desactualizado", axes[1])
fig.tight_layout()
plt.show()

**Para observar:** en CP, `C` no puede leer ni escribir mientras está aislado (pierde **disponibilidad**). En AP, `C` responde `100` aunque ya se había escrito `80` (pierde **consistencia**), y al reparar la red una de las dos escrituras concurrentes desaparece.

## PACELC: aun sin partición, hay que elegir entre latencia y consistencia

Sin cortes de red (la *E* de *Else*), una escritura puede confirmarse cuando responde **una** réplica (`W=1`, rápido) o esperar a **todas** (`W=3`, consistente). Simulamos 10.000 escrituras: las réplicas `A` y `B` están en la misma región (≈5 ms) y `C` está en otra región (+40 ms).

In [ ]:
rng = np.random.default_rng(7)
ops, n_replicas = 10_000, 3
latency = rng.lognormal(mean=np.log(5), sigma=0.5, size=(ops, n_replicas))   # ms por réplica
latency[:, 2] += 40                                                           # réplica remota

rows = []
for w in range(1, n_replicas + 1):
    ack = np.sort(latency, axis=1)[:, w - 1]      # se confirma cuando responde la W-ésima réplica
    rows.append({"W": w, "p50_ms": np.percentile(ack, 50), "p99_ms": np.percentile(ack, 99)})
lat = pd.DataFrame(rows)
display(lat.round(1))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(lat))
ax.bar(x - 0.2, lat["p50_ms"], 0.4, label="p50", color="#93c5fd")
ax.bar(x + 0.2, lat["p99_ms"], 0.4, label="p99", color="#1d4ed8")
ax.set_xticks(x, [f"W={w}" for w in lat["W"]])
ax.set_ylabel("Latencia de escritura (ms)")
ax.set_title("Más réplicas confirmadas = más consistencia, pero más latencia")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.show()

## Quórum: ¿cuándo una lectura ve siempre la última escritura?

Con `N` réplicas, una escritura se guarda en `W` réplicas y una lectura consulta `R` réplicas. Simulamos muchas combinaciones elegidas al azar y medimos con qué frecuencia la lectura **no** encuentra la versión nueva.

In [ ]:
def stale_read_probability(n, w, r, trials=20_000, seed=11):
    rng = np.random.default_rng(seed)
    stale = 0
    for _ in range(trials):
        written = set(rng.choice(n, size=w, replace=False))
        read = set(rng.choice(n, size=r, replace=False))
        stale += written.isdisjoint(read)
    return stale / trials

N = 3
quorum = pd.DataFrame([
    {"N": N, "W": w, "R": r, "W+R>N": w + r > N, "prob_lectura_vieja": stale_read_probability(N, w, r)}
    for w in range(1, N + 1) for r in range(1, N + 1)
])
display(quorum)

## Tu turno

1. Cambiá `N = 3` por `N = 5` en la celda anterior y volvé a ejecutarla. Anotá qué combinaciones de `W` y `R` tienen probabilidad 0 de lectura vieja.
2. Modificá `run_scenario` para usar cinco réplicas (`["A", "B", "C", "D", "E"]`) con la partición `{"A","B","C"}` y `{"D","E"}`. ¿Qué lado sigue aceptando escrituras en modo CP? ¿Por qué?

In [ ]:
# Tu solución (punto 2)